# Pipeline SDC — Analyse de métadonnées par LLM

Ce notebook présente le pipeline `metadata-analysis-llm-for-sdc` : il transforme un
classeur de métadonnées en un **tableau plat normalisé** pour rtauargus.

Run pip install -r requirements-notebook.txt

## Vue d'ensemble

| Module | Rôle |
|---|---|
| `read_input.py` | Classeur (.ods/.xlsx/.csv) → Markdown — **déterministe** |
| `llm_client.py` | Appel au modèle Qwen (API compatible OpenAI) |
| `pipeline.py` | Orchestrateur : sérialisation → Phase 1 → Phase 2 |
| `extract_json.py` | Valide le JSON du modèle contre le schéma |
| `json_to_table.py` | JSON validé → `.csv` (humain) |
| `run_full_pipeline_on_minio.py` | Pipeline complet MinIO → MinIO (script terminal) |

**Flux** :
```
classeur (MinIO)
  → serialize          déterministe  → Markdown
  → Phase 1 (LLM)     probabiliste  → questions au producteur  ← vous répondez ici
  → Phase 2 (LLM)     probabiliste  → JSON validé
  → write_csv          déterministe  → CSV (MinIO)
```


In [ ]:
# pour changer de modèle (autre que qwen dans ce cas)
import os 
os.environ["LLM_MODEL"] = "gemma4-26b-moe"

## 1. Mise en place

> **Prérequis Onyxia :** service VSCode-python lancé avec `CLE_API_OPENWEBUI` déclarée en variable d'environnement, et `pip install -r requirements.txt` exécuté dans le terminal. EDGE FONCTIONNE MIEUX QUE FIREFOX.

In [ ]:
import os, sys, tempfile
from pathlib import Path
import s3fs
from IPython.display import Markdown, display

# Détection de la racine du repo (fonctionne que le notebook soit lancé
# depuis la racine ou depuis notebooks/)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "core").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'core/'. Lancez Jupyter depuis la racine du repo.")

sys.path.insert(0, str(REPO_ROOT))
from core import pipeline, extract_json, json_to_table

# Vérification de l'environnement
api_key = os.environ.get("CLE_API_OPENWEBUI") or os.environ.get("OPENAI_API_KEY")
assert api_key, "Clé API introuvable — déclarez CLE_API_OPENWEBUI dans les variables d'environnement Onyxia."
print(f"✓ Clé API       : {'*' * (len(api_key))}")
print(f"✓ Modèle        : {os.environ.get('LLM_MODEL', 'qwen3-6-35b-moe')}")

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
)
print("✓ MinIO         : connexion initialisée")

## 2. Sérialisation — ce que le modèle reçoit

`read_input.serialize` convertit le classeur en Markdown pur.
Cette étape est **100 % déterministe** : même fichier → même Markdown.


In [ ]:
INPUT_S3  = "[path_here]"
OUTPUT_S3 = "[path_here]"

# Téléchargement depuis MinIO vers /tmp
tmp_input = Path(f"/tmp/input{Path(INPUT_S3).suffix}")
with fs.open(INPUT_S3, "rb") as f_in:
    tmp_input.write_bytes(f_in.read())
print(f"✓ Téléchargé : {INPUT_S3}\n")

# Sérialisation
md_input = pipeline.serialize(tmp_input)
print(f"  {len(md_input)} caractères — aperçu :")
display(Markdown(md_input))

## 3. Phase 1 — le modèle analyse et pose ses questions

`pipeline.start()` envoie le Markdown au modèle avec les instructions.

Le modèle produit :
- des **notes de travail** (avant le séparateur `---`)
- puis soit une **liste de questions** (si des ambiguïtés subsistent), soit `Aucune question.`
  suivi du JSON directement

> ⏳ Cette cellule appelle le LLM — comptez environ 1 à 3 minutes.


In [ ]:
print("Phase 1 — envoi au modèle...\n")
r = pipeline.start(md_input)

if r.auto_continued:
    print("Le modèle n'a posé aucune question — JSON produit directement.")
    print("Passez à la cellule Phase 2 (la cellule 'Réponses' sera ignorée).")
else:
    print("─" * 70)
    print(r.questions)
    print("─" * 70)
    print("\n→ Remplissez la variable `answers` dans la cellule ci-dessous, puis exécutez-la.")

## 4. Réponses du producteur

Remplissez la variable `answers` avec vos réponses aux questions ci-dessus
(une réponse par question, dans l'ordre), puis exécutez la cellule.


In [ ]:
# Remplissez vos réponses ici avant d'exécuter cette cellule.
answers = """

"""

print("Réponses enregistrées — exécutez la cellule Phase 2.")

## 5. Phase 2 — JSON, validation et CSV final

`pipeline.answer()` envoie vos réponses au modèle qui produit le JSON.
Le JSON est ensuite **validé contre le schéma** (étape déterministe), puis
converti en tableau plat et écrit sur MinIO.

> ⏳ Cette cellule appelle à nouveau le LLM — comptez environ 1 à 2 minutes.


In [ ]:
import pandas as pd

if r.auto_continued:
    # Phase 1 a déjà produit le JSON — pas besoin d'un second appel
    records = r.records
    print("Résultats issus de la Phase 1 (aucune question posée).\n")
else:
    print("Phase 2 — envoi des réponses au modèle...\n")
    records = pipeline.answer(r.history, answers)
    print(f"✓ {len(records)} enregistrement(s) validés contre le schéma.\n")

# Écriture du CSV sur MinIO
tmp_csv = Path("/tmp/output.csv")
cols, rows = json_to_table.write_csv(records, tmp_csv)

with open(tmp_csv, "r", encoding="utf-8-sig") as f:
    csv_content = f.read()
with fs.open(OUTPUT_S3, "w", encoding="utf-8") as f_out:
    f_out.write(csv_content)

print(f"✓ CSV écrit sur MinIO : {OUTPUT_S3}")
print(f"  {len(rows)} lignes × {len(cols)} colonnes\n")

# Affichage du résultat
df = pd.read_csv(tmp_csv, encoding="utf-8-sig", keep_default_na=False)
display(df)

## 6. Suivi des erreurs — `results.csv`

Compare la sortie du pipeline avec un **CSV de référence** (sortie attendue corrigée à la main)
et enregistre les résultats dans `notebook/results.csv`.

Chaque run ajoute **une ligne** avec :
- le nom du fichier d'entrée et l'horodatage UTC du run,
- le nombre total de cellules incorrectes,
- la décomposition par variable (`field`, `hrc_field`, `indicator`, `hrc_indicator`, variables de croisement).

Répétés sur plusieurs fichiers, ces chiffres permettent d'identifier si le LLM se trompe
**systématiquement** sur une variable précise (problème de prompt/modèle) plutôt qu'aléatoirement.

> **Avant d'exécuter** : renseignez `REFERENCE_CSV` avec le chemin vers votre CSV de référence
> (MinIO ou local). Le CSV de référence doit avoir la même structure que la sortie du pipeline.

In [ ]:
import re, csv
from datetime import datetime, timezone

# ── À remplir : chemin vers le CSV de référence (sortie attendue pour ce fichier) ──
REFERENCE_CSV = "[path_here]"   # MinIO (ex. user/data/ref.csv) ou chemin local
# ────────────────────────────────────────────────────────────────────────────────────

RESULTS_CSV  = REPO_ROOT / "notebook" / "results.csv"
RESULTS_HTML = REPO_ROOT / "notebook" / "results.html"

def _load_ref(path, fs):
    try:
        with fs.open(path, "r", encoding="utf-8-sig") as f:
            return pd.read_csv(f, keep_default_na=False)
    except Exception:
        return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)

ref = _load_ref(REFERENCE_CSV, fs)
out = df.copy()

# ── Comparaison du nombre de tableaux ───────────────────────────────────────
n_tables_ref  = len(ref)
n_tables_out  = len(out)
n_tables_diff = n_tables_out - n_tables_ref

# ── Alignement sur table_name (fallback positionnel) ────────────────────────
def _align(ref_df, out_df):
    if "table_name" in ref_df.columns and "table_name" in out_df.columns:
        ref_ids = ref_df["table_name"].tolist()
        out_lookup = set(out_df["table_name"])
        common = [t for t in ref_ids if t in out_lookup]
        if common:
            ra = ref_df.set_index("table_name").reindex(common)
            oa = out_df.set_index("table_name").reindex(common)
            return ra, oa, len(ref_df) - len(common), len(out_df) - len(common)
    n = min(len(ref_df), len(out_df))
    return (ref_df.iloc[:n].reset_index(drop=True),
            out_df.iloc[:n].reset_index(drop=True),
            len(ref_df) - n, len(out_df) - n)

ref_a, out_a, n_miss_ref, n_miss_out = _align(ref, out)

# Normalise les cellules vides en "NA" dans la référence (sentinel du schéma)
ref_a = ref_a.fillna("NA").replace("", "NA")
out_a = out_a.fillna("NA").replace("", "NA")

# ── Comptage des cellules incorrectes par colonne ────────────────────────────
SPAN_RE     = re.compile(r"^spanning_\d+$")
HRC_SPAN_RE = re.compile(r"^hrc_spanning_\d+$")

col_errors = {"table_name": 0, "field": 0, "hrc_field": 0,
              "indicator": 0, "hrc_indicator": 0,
              "spanning": 0, "hrc_spanning": 0}

for col in ref_a.columns:
    if col not in out_a.columns:
        continue
    n_wrong = int((ref_a[col].astype(str) != out_a[col].astype(str)).sum())
    if col in col_errors:
        col_errors[col] += n_wrong
    elif SPAN_RE.match(col):
        col_errors["spanning"] += n_wrong
    elif HRC_SPAN_RE.match(col):
        col_errors["hrc_spanning"] += n_wrong

unmatched_err = (n_miss_ref + n_miss_out) * max(len(ref_a.columns), 1)
total_wrong   = sum(col_errors.values()) + unmatched_err

# ── Résumé ───────────────────────────────────────────────────────────────────
diff_label = "=" if n_tables_diff == 0 else f"{n_tables_diff:+d}"
print(f"  Tableaux attendus  : {n_tables_ref}")
print(f"  Tableaux produits  : {n_tables_out}  [{diff_label}]")
if n_tables_diff != 0:
    print(f"  !! LLM a {'ajouté' if n_tables_diff > 0 else 'omis'} {abs(n_tables_diff)} tableau(x)")
print(f"\n  {'Colonne':<22} {'Cellules incorrectes':>20}")
print("  " + "─" * 44)
for k, v in col_errors.items():
    print(f"  {k:<22} {v:>20}")
if n_miss_ref or n_miss_out:
    print(f"\n  Lignes manquantes : {n_miss_ref}")
    print(f"  Lignes en excès   : {n_miss_out}")
print(f"\n  ➜  TOTAL : {total_wrong} cellules incorrectes")

# ── results.csv ──────────────────────────────────────────────────────────────
RESULT_COLS = [
    "filename", "timestamp", "total_wrong_cells",
    "n_tables_ref", "n_tables_out", "n_tables_diff",
    "table_name", "field", "hrc_field",
    "indicator", "hrc_indicator",
    "spanning", "hrc_spanning",
    "missing_from_output", "extra_in_output",
]
row = {
    "filename":            Path(INPUT_S3).name,
    "timestamp":           datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "total_wrong_cells":   total_wrong,
    "n_tables_ref":        n_tables_ref,
    "n_tables_out":        n_tables_out,
    "n_tables_diff":       n_tables_diff,
    **col_errors,
    "missing_from_output": n_miss_ref,
    "extra_in_output":     n_miss_out,
}

write_header = not RESULTS_CSV.exists()
with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=RESULT_COLS, extrasaction="ignore")
    if write_header:
        w.writeheader()
    w.writerow(row)

# ── results.html ─────────────────────────────────────────────────────────────
results_df_full = pd.read_csv(RESULTS_CSV)

html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="utf-8">
<title>Suivi des erreurs LLM</title>
<style>
  body {{ font-family: sans-serif; padding: 2em; background: #f9f9f9; }}
  h2   {{ color: #333; }}
  table {{ border-collapse: collapse; width: 100%; background: white;
           box-shadow: 0 1px 4px rgba(0,0,0,.1); }}
  th   {{ background: #3a6ea5; color: white; padding: 8px 12px;
           text-align: left; font-size: .85em; white-space: nowrap; }}
  td   {{ padding: 7px 12px; border-bottom: 1px solid #e0e0e0;
           font-size: .85em; white-space: nowrap; }}
  tr:hover td {{ background: #eef4ff; }}
  td.num  {{ text-align: right; font-variant-numeric: tabular-nums; }}
  td.hi   {{ background: #ffd6d6; font-weight: bold; }}
  td.warn {{ background: #fff3cd; font-weight: bold; }}
  p.ts    {{ color: #888; font-size: .8em; }}
</style>
</head>
<body>
<h2>Suivi des erreurs LLM — cellules incorrectes par variable</h2>
<p class="ts">Généré le {datetime.now().strftime("%Y-%m-%d %H:%M")} · {len(results_df_full)} run(s)</p>
<table>
<thead><tr>{"".join(f"<th>{c}</th>" for c in results_df_full.columns)}</tr></thead>
<tbody>
"""

NUM_COLS  = set(RESULT_COLS) - {"filename", "timestamp"}
DIFF_COLS = {"n_tables_diff"}

for _, r in results_df_full.iterrows():
    html += "<tr>"
    for col in results_df_full.columns:
        val = r[col]
        if col in DIFF_COLS:
            v = int(val) if not pd.isna(val) else 0
            html += f'<td class="num warn">{v:+d}</td>' if v != 0 else f'<td class="num">{v}</td>'
        elif col in NUM_COLS and isinstance(val, (int, float)) and val > 0:
            html += f'<td class="num hi">{int(val)}</td>'
        elif col in NUM_COLS:
            html += f'<td class="num">{int(val) if not pd.isna(val) else 0}</td>'
        else:
            html += f"<td>{val}</td>"
    html += "</tr>\n"

html += "</tbody></table></body></html>"
RESULTS_HTML.write_text(html, encoding="utf-8")

print(f"\n✓ CSV  → {RESULTS_CSV.relative_to(REPO_ROOT)}")
print(f"✓ HTML → {RESULTS_HTML.relative_to(REPO_ROOT)}")